In [12]:
import json
import pandas as pd


In [32]:
import json
import pandas as pd

def check_session_completeness_by_process_text(session_id, json_data, xlsx_path):

    session_key, block_str = session_id.split("_S")
    block_index = int(block_str) - 1

    if session_key not in json_data:
        print(f"⚠️ Session key '{session_key}' nicht gefunden.")
        return

    try:
        block = json_data[session_key][block_index]
    except IndexError:
        print(f"⚠️ Kein Block {block_index} in Session '{session_key}'.")
        return

    # Prozess-Texte aus JSON holen
    json_processes = {p.get("process", "").strip() for p in block}

    # Sheet-Name = session_id
    sheet_to_use = session_id

    # Excel-Datei lesen
    try:
        df = pd.read_excel(xlsx_path, sheet_name=sheet_to_use, usecols="B", skiprows=2, header=None)
    except ValueError:
        print(f"⚠️ Sheet '{sheet_to_use}' nicht gefunden in Datei.")
        return
    except Exception as e:
        print(f"⚠️ Fehler beim Einlesen: {e}")
        return

    excel_processes = set(df[1].dropna().astype(str).str.strip())

    matched = json_processes & excel_processes
    unmatched = json_processes - matched
    completeness_percent = (len(matched) / len(json_processes)) * 100 if json_processes else 0

    # Ausgabe
    print(f"\n=== Completeness Check: {session_id} ===")
    print(f"Prozesse in JSON: {len(json_processes)}")
    print(f"In Excel gefunden: {len(matched)}")
    print(f"Fehlend: {len(unmatched)}")
    print(f"✅ Completeness: {round(completeness_percent, 2)}%")

    if unmatched:
        print("\n❌ Fehlende Prozesse:")
        for val in unmatched:
            print(f" - {val}")

    return {
        "session_id": session_id,
        "total_in_json": len(json_processes),
        "found_in_excel": len(matched),
        "missing": list(unmatched),
        "completeness_percent": round(completeness_percent, 2)
    }


In [59]:
path_pre_processed = "data/pre_processed_modified_pbt_annotation_of_mini_session.json"
path_no_processed = "modified_pbt_annotation_of_mini_session (1).json"

with open(path_pre_processed) as f:
    data = json.load(f)

ev3_path = "Evaluation_Sheets/Evaluated Sheets/First Round/EV3_Evaluation_07_02_25.xlsx"
ev1_path = "Evaluation_Sheets/Evaluated Sheets/First Round/EV1_Evaluation_07_02_25.xlsx"
ev2_path = "Evaluation_Sheets/Evaluated Sheets/First Round/EV2_Evaluation_07_02_25.xlsx"

check_session_completeness_by_process_text("P16_S01",data, ev3_path)


=== Completeness Check: P16_S01 ===
Prozesse in JSON: 14
In Excel gefunden: 10
Fehlend: 4
✅ Completeness: 71.43%

❌ Fehlende Prozesse:
 - And then that leads to blaming itself and negative self talk.
 - I would say worthlessness.
 - I would say worthlessness. And it's kind of a bi-directional relationship there too.
 - Yeah. So if struggle to do good and change behavior are similar, then procrastination and blame body images and blame and worthlessness are kind of connected here too?


{'session_id': 'P16_S01',
 'total_in_json': 14,
 'found_in_excel': 10,
 'missing': ['And then that leads to blaming itself and negative self talk.',
  'I would say worthlessness.',
  "I would say worthlessness. And it's kind of a bi-directional relationship there too.",
  'Yeah. So if struggle to do good and change behavior are similar, then procrastination and blame body images and blame and worthlessness are kind of connected here too?'],
 'completeness_percent': 71.43}